# 08 · Proyección experimental 2027 actualizada

**Objetivo.** Construir una proyección seccional auditable para 2027 usando sólo información electoral hasta 2024, manteniendo separados el baseline, la corrección aprendida por árboles, el ajuste nacional y la incertidumbre.

**Baseline.** Participación 2021 de la misma sección o componente territorial oficial. Para C/D se usa la jerarquía sección→componente→municipio→entidad→nacional. En los backtests este baseline A+B tuvo MAE ponderado de 9.38 pp en 2021 y 4.42 pp en 2024.

**Modelo seleccionado.** `HistGradientBoostingRegressor` con `territorial_context`, ganador de la comparación controlada contra XGBoost, LightGBM y Extra Trees. Predice el residual sobre el baseline; sólo se aplica 25% de esa corrección. Después se suma, por separado, el desplazamiento nacional de +5.21 pp estimado con el ciclo intermedio 2015→2021.

**Corte.** Elecciones hasta 2024; geografía observada en el snapshot 2024; linajes públicos AMGE disponibles hasta 2022. No existe todavía marco seccional ni resultado 2027.

**Salidas.** Feature mart 2027, predicciones A/B por unidad, expansión a 70,751 secciones del snapshot, intervalos por clase, resumen, registro del modelo y decisión experimental.

## Configuración y corte informativo

Se valida que ninguna elección posterior a 2024 entre al entrenamiento o a las variables. El target 2027 no aparece como outcome en ningún artefacto.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import yaml
from IPython.display import Markdown, display

from radar27.modeling.forecast_2027 import (
    build_forecast_feature_mart,
    expand_to_section_forecast,
    train_and_predict_2027,
    write_forecast_decision,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs/forecast_2027.yaml"
        ).exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
with (PROJECT_ROOT / "configs/forecast_2027.yaml").open(encoding="utf-8") as stream:
    forecast_config = yaml.safe_load(stream)
required_inputs = [
    PROJECT_ROOT / "artifacts/tables/feature_ablation_mart.parquet",
    PROJECT_ROOT / "artifacts/tables/national_shift_estimates.csv",
    PROJECT_ROOT / "artifacts/tables/tabular_challenger_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/tabular_challenger_metrics.csv",
    PROJECT_ROOT / "artifacts/tabular_challenger_decision.json",
]
input_checks = {path.relative_to(PROJECT_ROOT).as_posix(): path.exists() for path in required_inputs}
display(pd.Series(input_checks, name="existe").to_frame())
display(pd.Series(forecast_config, name="valor").to_frame())
assert all(input_checks.values())

## Feature mart operacional enriquecido

Las unidades A son secciones estables; las B son componentes de linaje oficial. Las variables terminan en 2024 y el rezago de misma familia es 2021. Se añaden promedios y diferencias relativas frente a la entidad y al país, calculados sólo con rezagos.

In [ ]:
forecast_features = build_forecast_feature_mart(PROJECT_ROOT)
feature_checks = {
    "unit_id_unique": forecast_features.evaluation_unit_id.is_unique,
    "only_A_B": set(forecast_features.geography_confidence_class) == {"A", "B"},
    "all_features_before_2027": bool(
        (forecast_features.max_electoral_feature_year < 2027).all()
    ),
    "no_target_outcome": "actual" not in forecast_features,
    "territorial_context_available": all(
        column in forecast_features
        for column in [
            "entity_same_type_mean", "same_type_minus_entity_mean",
            "national_same_type_mean", "geography_unit_type",
        ]
    ),
    "context_is_prior": bool(
        (forecast_features.context_source_max_year < 2027).all()
    ),
}
display(pd.Series(feature_checks, name="aprobado").to_frame())
display(forecast_features.groupby("geography_confidence_class").agg(
    units=("evaluation_unit_id", "size"),
    sections=("target_sections_represented", "sum"),
))
assert all(feature_checks.values())

## Entrenamiento del boosting residual

El modelo no predice desde cero cuando existe baseline. Los árboles estiman `participación real − participación de la elección anterior del mismo tipo`; se aplica 25% de esa corrección para limitar sobreajuste. El ajuste nacional permanece en otra columna.

In [ ]:
model_units, model_registry = train_and_predict_2027(
    PROJECT_ROOT, forecast_features
)
display(pd.Series({
    "model_id": model_registry["model_id"],
    "family": model_registry["model_family"],
    "feature_group": model_registry["feature_set"],
    "training_years": model_registry["training_target_years"],
    "training_units": model_registry["training_units"],
    "residual_shrinkage": model_registry["residual_shrinkage"],
    "national_shift_pp": 100 * model_registry["national_shift_correction"],
}).to_frame("valor"))
display(pd.Series(model_registry["hyperparameters"]).to_frame("valor"))

In [ ]:
model_checks = {
    "training_stops_in_2024": model_registry["maximum_training_target_year"] == 2024,
    "selected_model_id": model_registry["model_id"] == "hist_gradient_boosting",
    "selected_feature_group": model_registry["feature_set"] == "territorial_context",
    "frozen_shrinkage": model_registry["residual_shrinkage"] == 0.25,
    "predictions_are_probabilities": bool(
        model_units.experimental_prediction.between(0, 1).all()
    ),
}
display(pd.Series(model_checks, name="aprobado").to_frame())
assert all(model_checks.values())

## Cobertura territorial e incertidumbre

El modelo se aplica a A/B. Sus radios se recalibran con errores fuera de tiempo 2021 del modelo territorial. Las clases C/D reciben el baseline jerárquico más la corrección nacional y radios de errores C/D de 2021. La expansión usa el snapshot 2024 y no afirma conocer cambios territoriales futuros.

In [ ]:
section_forecast, forecast_summary = expand_to_section_forecast(
    PROJECT_ROOT, model_units
)
display(forecast_summary[
    forecast_summary.group_dimension.isin([
        "overall", "geography_confidence_class", "prediction_strategy"
    ])
])
display(pd.read_csv(PROJECT_ROOT / "artifacts/tables/forecast_2027_interval_calibration.csv"))

In [ ]:
section_checks = {
    "snapshot_sections_unique": section_forecast.section_key_observed.is_unique,
    "snapshot_section_count": len(section_forecast) == 70_751,
    "all_classes_present": set(section_forecast.geography_confidence_class) == {"A", "B", "C", "D"},
    "all_sections_predicted": bool(section_forecast.experimental_prediction.notna().all()),
    "intervals_ordered": bool(
        (section_forecast.lower <= section_forecast.experimental_prediction).all()
        and (section_forecast.experimental_prediction <= section_forecast.upper).all()
    ),
    "snapshot_not_future_geography": section_forecast.forecast_geography_status.eq(
        "2024_snapshot_not_2027_certified"
    ).all(),
}
display(pd.Series(section_checks, name="aprobado").to_frame())
assert all(section_checks.values())

## Comparación histórica y decisión

No existe MAE 2027 porque aún no hay resultado real. La referencia disponible son backtests históricos contra el baseline de última elección de la misma familia.

In [ ]:
forecast_decision = write_forecast_decision(
    PROJECT_ROOT, model_units, section_forecast, forecast_summary, model_registry
)
historical_comparison = pd.DataFrame(forecast_decision["historical_mae_comparison"])
historical_comparison["baseline_mae_pp"] = 100 * historical_comparison["baseline_weighted_mae"]
historical_comparison["corrected_model_mae_pp"] = 100 * historical_comparison[
    "corrected_boosting_weighted_mae"
]
historical_comparison["change_pp"] = 100 * historical_comparison["mae_change_vs_baseline"]
display(historical_comparison[[
    "target_year", "baseline_model", "baseline_mae_pp",
    "corrected_model_mae_pp", "change_pp",
]])
display(pd.Series(forecast_decision).to_frame("valor"))

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/forecast_2027_experimental.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join([
    "<h1>RADAR 27 · Proyección experimental 2027</h1>",
    "<p>Generado por notebooks/08_forecast_2027_experimental.ipynb.</p>",
    "<h2>Modelo</h2><pre>" + json.dumps(model_registry, ensure_ascii=False, indent=2) + "</pre>",
    "<h2>Comparación histórica</h2>", historical_comparison.to_html(index=False),
    "<h2>Resumen de proyección</h2>", forecast_summary.to_html(index=False),
    "<h2>Decisión</h2><pre>" + json.dumps(
        forecast_decision, ensure_ascii=False, indent=2
    ) + "</pre>",
]), encoding="utf-8")

required_outputs = [
    PROJECT_ROOT / "artifacts/tables/forecast_2027_feature_mart.parquet",
    PROJECT_ROOT / "artifacts/tables/forecast_2027_model_units.parquet",
    PROJECT_ROOT / "artifacts/tables/forecast_2027_section_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/forecast_2027_interval_calibration.csv",
    PROJECT_ROOT / "artifacts/tables/forecast_2027_summary.csv",
    PROJECT_ROOT / "artifacts/forecast_2027_model_registry.json",
    PROJECT_ROOT / "artifacts/forecast_2027_decision.json",
    report_path,
]
final_checks = {
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    **feature_checks,
    **model_checks,
    **section_checks,
}
display(pd.Series(final_checks, name="aprobado").to_frame())
assert all(final_checks.values())

In [ ]:
overall = forecast_summary[
    forecast_summary.group_dimension.eq("overall")
].iloc[0]
display(Markdown(
    f"**Cierre experimental.** Se proyectaron `{len(section_forecast):,}` secciones del snapshot 2024. "
    f"Promedio ponderado por lista nominal actual: "
    f"`{100*overall.weighted_experimental_prediction:.2f}%`. "
    "No es resultado oficial, estimación de votos ni garantía pública. "
    f"Tiempo total: `{time.perf_counter()-RUN_STARTED:.1f}` s."
))